# TRIBE v2 — Cognitive Load Demo on Merge Conflicts

Demonstração mínima de uso do TRIBE v2 (Meta, 2026) para estimar carga cognitiva induzida por blocos de conflito de merge.

**Como rodar no Colab Free:**
1. Upload da pasta `tribe/` inteira (via painel lateral → Files → Upload folder).
2. Runtime → Change runtime type → GPU (T4).
3. Executar células em ordem.

**Hipótese:** `scenario_38` (paradigm clash) deve apresentar carga maior que `01-method-conflict` (conflito textual trivial).

## 1. Setup — clonar o repo e instalar

In [ ]:
!git clone https://github.com/facebookresearch/tribev2.git
%cd tribev2
!pip install -q -e ".[plotting]"
%cd ..

In [ ]:
# Verifica GPU
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 2. Carregar o modelo

Primeira execução baixa os pesos (alguns GB). Subsequentes usam cache local.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tribev2 import TribeModel

model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
print("Model loaded.")

## 3. Listar conflitos de teste

In [ ]:
conflict_dir = Path("conflicts")
conflicts = sorted(conflict_dir.glob("*.txt"))
print(f"Found {len(conflicts)} conflict file(s):")
for c in conflicts:
    print(f"  - {c.name} ({c.stat().st_size} bytes)")

## 4. Predição de carga cognitiva por conflito

Para cada conflito, o TRIBE converte o texto em fala internamente, processa pelos encoders (LLaMA 3.2, Wav2Vec-BERT) e prediz a resposta fMRI na malha cortical fsaverage5 (~20k vértices).

**Proxy de carga (versão crua):**
- `mean_load` = magnitude média da ativação predita em todos os vértices e timesteps.
- `peak_load` = maior média de ativação em um único timestep (pico cognitivo).

Versão refinada (futura): mascarar pela rede frontoparietal (Yeo-7) antes de agregar.

In [ ]:
results = []
for path in conflicts:
    print(f"\nProcessing {path.name}...")
    df_events = model.get_events_dataframe(text_path=str(path))
    preds, segments = model.predict(events=df_events)
    mean_load = float(np.mean(np.abs(preds)))
    peak_load = float(np.max(np.mean(np.abs(preds), axis=1)))
    results.append({
        "conflict": path.stem,
        "timesteps": int(preds.shape[0]),
        "vertices": int(preds.shape[1]),
        "mean_load": mean_load,
        "peak_load": peak_load,
    })
    print(f"  shape: {preds.shape}")
    print(f"  mean_load: {mean_load:.4f}")
    print(f"  peak_load: {peak_load:.4f}")

## 5. Comparar resultados

In [ ]:
df_results = pd.DataFrame(results).sort_values("mean_load", ascending=False).reset_index(drop=True)
df_results

In [ ]:
df_results.to_csv("tribe_results.csv", index=False)
print("Saved to tribe_results.csv — faça download via painel lateral.")

## 6. Interpretação

**Se a hipótese for confirmada** (`scenario_38` > `01-method-conflict`): TRIBE diferencia dificuldade neurocognitiva entre tipos de conflito. Justifica seguir com o desenho de roteamento adaptativo (3 pilares ↔ estados).

**Se inconclusivo ou invertido**: investigar:
- Tamanho do estímulo (scenario_38 é muito maior — normalizar por timesteps).
- Agregação global mascara sinais específicos — testar ROIs frontoparietais.
- Conversão texto→fala pode descaracterizar código — considerar input visual (renderização do diff como imagem) via branch de vídeo do TRIBE.

**Próximos passos:**
1. Adicionar mais conflitos em `conflicts/` (cobrir os 3 modos de falha do `access.tex`).
2. Normalizar carga por tamanho do estímulo.
3. Implementar máscara frontoparietal (Yeo-7) para score principled.